# Experiment 1.3.9 — Phase-aware Cross-user Contrastive Local Feature Learning

## Scientific question

Experiments 1.3.7–1.3.8 showed that the SNN local representation already contains task-relevant information complementary to Raw250 counts, while directly reconstructing raw count or early/late input statistics does not improve the SNN-only local representation.

This experiment therefore stops reconstructing raw input statistics and instead shapes the **geometry of the deployed SNN representation itself**. The hypothesis is that local SNN features should become more stable across users when local motion motifs with the same letter label and similar relative gesture phase are pulled together, while different letters in the same phase are separated.

The final deployment target remains SNN-only:

\[X_{spike} \rightarrow SNN \rightarrow z_1,z_2,\ldots,z_B, \qquad z_b\in\mathbb{R}^{64}.\]

No Raw branch and no projection head are kept at inference.

## Experimental conditions

- **A — `cls_only`**: whole-gesture CE only.
- **B — `con250`**: CE + phase-aware cross-user supervised contrastive loss directly on each 250 ms SNN local feature.
- **C — `con500`**: CE + the same contrastive loss on a 500 ms motif formed by concatenating two adjacent 250 ms local features. The deployed feature is still the original 250 ms 64-D feature.

Positive pairs are restricted to **same label + same relative-phase bucket + different user**. Negatives are restricted to **different label + same phase bucket**. Anchors without both a positive and a negative do not contribute to the contrastive loss. Only fully-valid windows participate; padded/partial windows are excluded.

For `con250`, the contrasted representation is `L2Normalize(log1p(z_b))`. For `con500`, it is `L2Normalize(log1p(concat[z_b,z_{b+1}]))`. There is deliberately **no projection head**, so the contrastive gradient shapes the representation that is actually deployed.

## Hyperparameter selection and leakage control

Temperature is fixed to `0.1`. The contrastive weight is selected from `0.01, 0.03, 0.10`. For each contrastive condition, all three training seeds `(11, 23, 101)` are run for every candidate lambda. The selected lambda maximizes **mean validation SNN250 fresh-probe balanced accuracy across the three seeds**. The test split is not accessed during this development stage.

Only after lambda selection do the final runs evaluate test performance.

In [ ]:
%run ../scripts/experiment_1_3_9_phase_aware_contrastive/01_setup.py

## Model and loss

The SNN architecture is unchanged from the previous local-feature experiments: `30 → 128 → 128 → 64`, with shifts `((2,3),(2,3),(2))`, `tau_mem=22.54 ms`, threshold `0.5`, continuous state through the full gesture, and no reset at 250 ms boundaries.

The total objective is

\[L = L_{cls} + \lambda_{con} L_{SupCon}.\]

The global classifier still reads the flattened sequence of 250 ms local features. The contrastive term is an auxiliary training regularizer only.

In [ ]:
%run ../scripts/experiment_1_3_9_phase_aware_contrastive/02_model_training.py

## Development sweep and final confirmation

Primary representation quality is measured with a frozen SNN and a fresh train-only-standardized Logistic Regression probe. `C` is selected on validation from `1e-3, 1e-2, 1e-1, 1, 10`.

Primary readout: **SNN250**. Secondary readout: **SNN125**, extracted from the same frozen L3 spike train to test whether improvements are representation-level rather than tied to one readout resolution.

In [ ]:
%run ../scripts/experiment_1_3_9_phase_aware_contrastive/03_run_experiment.py

## Diagnostics

Two diagnostics are added without changing the target deployment architecture.

1. **Cross-user phase-aware retrieval/kNN**: query and candidate pool are constrained to different users and the same phase. This directly tests whether the intended cross-user geometry was created.
2. **Raw250 + SNN250 fusion**: diagnostic only. If SNN-only improves while fusion remains above Raw250, the representation became more robust without simply collapsing into a Raw-count copy.

Firing rates, dead-neuron fractions, and the fraction of contrastive-valid anchors are retained to diagnose pathological regimes.

In [ ]:
%run ../scripts/experiment_1_3_9_phase_aware_contrastive/04_diagnostics.py

## Decision rules

- If `con250` improves mean SNN250 BA and reduces seed variance, a 250 ms local feature is already a stable task-aware unit.
- If `con500` is stronger, individual 250 ms features likely act as primitives that become class-relevant only when composed into a short local motif.
- If both contrastive conditions underperform `cls_only`, letter identity is probably too strong a supervisory signal at this local timescale; the next step should move toward predictive/self-supervised local objectives rather than stronger class-conditioned local losses.
- Raw+SNN fusion is never treated as the final architecture; it is used only to diagnose retained complementarity.